# 2023년 의료기관별 시군구별 진료비 분석

## 단계: 01. 데이터 전처리 (Data Preprocessing)
- 목표: 원본 데이터를 분석 가능한 형태의 정제된 데이터셋으로 변환

### 1.1 환경 설정 및 데이터 로드
- 공공데이터포털에서 내려받은 CSV 파일을 판다스로 로드합니다.
- 한글 인코딩(cp949) 문제를 해결합니다.

In [1]:
import pandas as pd
import numpy as np

In [2]:
# 파일 경로 설정
filepath = r'C:\data\hira_sigungu_2023.csv'

In [3]:
# cp949 인코딩으로 불러오기
df = pd.read_csv(filepath, encoding='cp949')

In [4]:
# 데이터 크기 확인
print(f'행 : {df.shape[0]}, 열 : {df.shape[1]}')
print(df.head(3))

행 : 251, 열 : 8
   진료년도  시도  시군구      환자수   명세서청구건수     입내원일수   보험자부담금(선별포함)  요양급여비용총액(선별포함)
0  2023  서울  강남구  3182688  18499074  20105055  2302379610700   2962352500080
1  2023  서울  강동구  1060832  10934670  12123283   835458119590   1108786307310
2  2023  서울  강서구  1163258  11007256  11812064   699996239970    946311204440


### 1.2 데이터 구조 및 타입 파악
- 데이터의 컬럼 구성과 각 변수의 데이터 타입이 분석에 적절한지 확인합니다.

In [10]:
# 전체 구조 및 데이터 타입 확인
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 251 entries, 0 to 250
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   진료년도            251 non-null    int64 
 1   시도              251 non-null    object
 2   시군구             251 non-null    object
 3   환자수             251 non-null    int64 
 4   명세서청구건수         251 non-null    int64 
 5   입내원일수           251 non-null    int64 
 6   보험자부담금(선별포함)    251 non-null    int64 
 7   요양급여비용총액(선별포함)  251 non-null    int64 
dtypes: int64(6), object(2)
memory usage: 15.8+ KB
None


In [11]:
# 시도별 고유값 확인 (17개 광역자치단체 정상 여부)
print('\n시도 종류 : ', df['시도'].unique())


시도 종류 :  ['서울' '부산' '인천' '대구' '광주' '대전' '울산' '경기' '강원' '충북' '충남' '전북' '전남' '경북'
 '경남' '제주' '세종']


### 1.3 데이터 정제 (결측치, 중복값, 공백 확인)
- 데이터의 품질을 저해하는 요소를 체크하고 제거하는 과정입니다.

In [13]:
# 결측치 확인
print('결측치 개수 : \n', df.isnull().sum())

결측치 개수 : 
 진료년도              0
시도                0
시군구               0
환자수               0
명세서청구건수           0
입내원일수             0
보험자부담금(선별포함)      0
요양급여비용총액(선별포함)    0
dtype: int64


In [14]:
# 중복 행 확인 (시도 - 시군구 쌍이 중복되는지 확인)
print('\n중복 시군구 : ', df[['시도', '시군구']].duplicated().sum())


중복 시군구 :  0


In [15]:
# 숫자 뒤 공백 확인 (데이터 내 불필요한 공백 확인)
print('공백 포함 여부 : ', df['환자수'].astype(str).str.contains(' ').sum())

공백 포함 여부 :  0


### 1.4 변수 생성 및 단위 변환
- 분석의 편의성을 위해 단위를 조정하거나 새로운 지표를 생성합니다.

In [6]:
# 단위 : 원 -> 억 원으로 변환
df_view = df.copy()
df_view['요양급여비용총액(억원)'] = (df['요양급여비용총액(선별포함)'] / 1e8).round(1)
print(df_view[['시도', '시군구', '요양급여비용총액(억원)']].describe())

       요양급여비용총액(억원)
count    251.000000
mean    3496.406773
std     4105.430638
min       31.400000
25%      494.000000
50%     2330.200000
75%     5112.950000
max    29623.500000


### 1.5 기초 통계량 및 이상치 탐색
- 수치 데이터를 요약하고, 분포가 왜곡되었는지 확인합니다.

In [19]:
# 수치형 데이터 요약 통계
stats = df_view[['요양급여비용총액(억원)', '환자수', '입내원일수']].describe()
print(stats)

       요양급여비용총액(억원)           환자수         입내원일수
count    251.000000  2.510000e+02  2.510000e+02
mean    3496.406773  4.078571e+05  4.320838e+06
std     4105.430638  4.085043e+05  3.743635e+06
min       31.400000  9.264000e+03  6.360700e+04
25%      494.000000  6.908100e+04  9.990605e+05
50%     2330.200000  3.235060e+05  3.695387e+06
75%     5112.950000  6.277475e+05  6.697585e+06
max    29623.500000  3.182688e+06  2.010506e+07


> **평균(3,496억) > 중앙값(2,330억)** - 오른쪽 꼬리 분포<br>
> 소수의 대도시 시군구(강남구,서초구 등)가 평균을 끌어올리고 있다. 이 패턴은 총액 그대로 비교하면 왜곡이 생긴다. 1인당 진료비로 변환해야하는 이유이다.<br>

> **결측치 0개, 중복 0개** - 전처리 완료 조건 충족 <br>
> 수치형 6개 변수 모두 int64로 정상 인식, 결측 및 중복 없음. 파생변수 생성 후 바로 EDA 가능.<br>

> **시도 17개 정상 확인** <br>
> 서울,부산,인천,대구,광주,대전,울산,경기,강원,충북,충남,전북,전남,경북,경남,제주,세종. 세조싱가 1개 시군구로 포함되어 있음.

> **최소값(31억), 최대값(2조 9,623억)** - 900배의 격차 <br>
> 인구수의 차이인지, 그 지역에 대형상급종합병원이 몰려있는지 확인해 볼 수 있음.<br>

> **표준편차(4,105억) > 평균(3,496억)** <br>
> 표준편차가 평균보다 더 크다. 데이터의 변동성이 매우 심하다. '전국 평균'보다 '지역 유형별'로 나누어 분석하는 것이 의미 있을 수 있음.
>
> **Max (29,623억) > 75% (5,112억)** <br>
> 75% 지점보다 5배 이상 큰 값이 존재함. 상위 몇 개의 지역은 통계적으로 명백한 이상치 혹은 특이값에 해당할 수 있음. 하지만 의료 데이터에서는 틀린 데이터가 아닌, 핵심 분석 대상이 될 수 있음.

In [3]:
# 전처리를 마치며 느낀 점 (학습 기록)

# 1. 평균과 중앙값의 차이가 꽤 큽니다. 
#    - 소수의 지역이 전체 평균을 높이고 있는 것 같아서, 나중에 '1인당 진료비'로 계산해볼 필요가 있을 것 같습니다.
# 2. 지역별 편차가 생각보다 큽니다. 
#    - 최소값과 최대값이 900배나 차이가 나는데, 인구수 때문인지 대형 병원 유무 때문인지 궁금해집니다.
# 3. 데이터 정제 확인
#    - 다행히 결측치나 중복값이 없어서 바로 다음 단계로 넘어갈 수 있을 것 같습니다.